In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('trip_sample.csv')

In [4]:
df.columns

Index(['trip_id', 'mode', 'distance_km', 'fare_cad', 'wait_min', 'rating',
       'peak'],
      dtype='object')

In [5]:
# First-pass inspection: learn the shape before the story

df.shape
df.columns
df.head(10)
df.tail()
df.info()
df.dtypes

# quick missing-value count
df.isna().sum()

# unique values in a category
df['mode'].value_counts(dropna=False)


(6, 7)

Index(['trip_id', 'mode', 'distance_km', 'fare_cad', 'wait_min', 'rating',
       'peak'],
      dtype='object')

,trip_id,mode,distance_km,fare_cad,wait_min,rating,peak
0,T001,Bike,3.2,0.00,4.0,5,Yes
1,T002,Taxi,8.4,19.50,6.0,4,No
2,T003,Bus,6.1,3.75,9.0,3,Yes
3,T004,Taxi,2.7,11.20,12.0,2,Yes
4,T005,Bus,14.3,3.75,18.0,2,Yes
5,T006,Taxi,5.1,13.20,NaN,4,No


,trip_id,mode,distance_km,fare_cad,wait_min,rating,peak
1,T002,Taxi,8.4,19.50,6.0,4,No
2,T003,Bus,6.1,3.75,9.0,3,Yes
3,T004,Taxi,2.7,11.20,12.0,2,Yes
4,T005,Bus,14.3,3.75,18.0,2,Yes
5,T006,Taxi,5.1,13.20,NaN,4,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   trip_id      6 non-null      object 
 1   mode         6 non-null      object 
 2   distance_km  6 non-null      float64
 3   fare_cad     6 non-null      float64
 4   wait_min     5 non-null      float64
 5   rating       6 non-null      int64  
 6   peak         6 non-null      object 
dtypes: float64(3), int64(1), object(3)
memory usage: 468.0+ bytes


trip_id         object
mode            object
distance_km    float64
fare_cad       float64
wait_min       float64
rating           int64
peak            object
dtype: object

trip_id        0
mode           0
distance_km    0
fare_cad       0
wait_min       1
rating         0
peak           0
dtype: int64

mode
Taxi    3
Bus     2
Bike    1
Name: count, dtype: int64

# Data Aggregation (GROUP BY), can be single or multiple variables

In [6]:
# SQL: SELECT COUNT(*) AS n, AVG(wait_min) AS avg_wait 
# FROM DS 
# GROUP BY mode

# Named Aggregation

df.groupby("mode")["wait_min"].agg(
    n="count",
    avg_wait="mean"
).reset_index()

,mode,n,avg_wait
0,Bike,1,4.0
1,Bus,2,13.5
2,Taxi,2,9.0


In [16]:
# Named Aggregation
# aggregate one single column/field ("wait_min")

df.groupby("mode").agg(
    n=("wait_min", "count"),
    avg_wait=("wait_min", "mean")
).reset_index()

,mode,n,avg_wait
0,Bike,1,4.0
1,Bus,2,13.5
2,Taxi,2,9.0


In [6]:
# SQL: SELECT COUNT(*), AVG(wait_min), MIN(wait_min), MAX(wait_min)
# FROM DS GROUP BY mode 
# ORDER BY AVG(wait_min) 

# No-Named Aggregation

summary = (
    df.groupby('mode')['wait_min']
      .agg(['count', 'mean', 'median', 'min', 'max'])
      .sort_values('mean', ascending=False)
)


In [8]:
summary

,count,mean,median,min,max
mode,,,,,
Bus,2,13.5,13.5,9.0,18.0
Taxi,2,9.0,9.0,6.0,12.0
Bike,1,4.0,4.0,4.0,4.0


In [9]:
summary.reset_index()

,mode,count,mean,median,min,max
0,Bus,2,13.5,13.5,9.0,18.0
1,Taxi,2,9.0,9.0,6.0,12.0
2,Bike,1,4.0,4.0,4.0,4.0


# Pivot tables and crosstabs: summarize categories


In [35]:
# crosstabs: compute the frequency (ratio)

In [13]:
# compute the frequency

# SELECT mode,   SUM(CASE WHEN peak = 'No'  THEN 1 ELSE 0 END) AS No,
#     SUM(CASE WHEN peak = 'Yes' THEN 1 ELSE 0 END) AS Yes
# FROM trips  GROUP BY mode;

pd.crosstab(
    df['mode'],
    df['peak']
)

# show all

# SELECT mode,   SUM(CASE WHEN peak = 'No'  THEN 1 ELSE 0 END) AS No,
#     SUM(CASE WHEN peak = 'Yes' THEN 1 ELSE 0 END) AS Yes,  COUNT(*) AS All
# FROM trips  GROUP BY mode;

pd.crosstab(
    df['mode'],
    df['peak'], 
    margins=True
)

# compute the ratio (row)

# SELECT    mode,    SUM(CASE WHEN peak = 'No'  THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS No,
#     SUM(CASE WHEN peak = 'Yes' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS Yes
# FROM trips  GROUP BY mode;


pd.crosstab(
    df['mode'],
    df['peak'],
    normalize='index'
)

# compute the ratio (column)
# SELECT    mode,    SUM(CASE WHEN peak = 'No' THEN 1 ELSE 0 END) * 1.0 / SUM(SUM(CASE WHEN peak = 'No' THEN 1 ELSE 0 END)) OVER () AS No,
#     SUM(CASE WHEN peak = 'Yes' THEN 1 ELSE 0 END) * 1.0  / SUM(SUM(CASE WHEN peak = 'Yes' THEN 1 ELSE 0 END)) OVER () AS Yes
# FROM trips     GROUP BY mode;


pd.crosstab(
    df['mode'],
    df['peak'],
    normalize='columns'
)

peak,No,Yes
mode,,
Bike,0,1
Bus,0,2
Taxi,2,1


peak,No,Yes,All
mode,,,
Bike,0,1,1
Bus,0,2,2
Taxi,2,1,3
All,2,4,6


peak,No,Yes
mode,,
Bike,0.000000,1.000000
Bus,0.000000,1.000000
Taxi,0.666667,0.333333


peak,No,Yes
mode,,
Bike,0.0,0.25
Bus,0.0,0.50
Taxi,1.0,0.25


In [40]:
# Pivot tables

# SELECT  mode,    AVG(CASE WHEN peak = 'No'  THEN fare END) AS No,  AVG(CASE WHEN peak = 'Yes' THEN fare END) AS Yes
# FROM trips   GROUP BY mode;


pd.pivot_table(df,
    index='mode',
    columns='peak',
    values='fare_cad',
    aggfunc='mean'
)


peak,No,Yes
mode,,
Bike,NaN,0.00
Bus,NaN,3.75
Taxi,16.35,11.20


# Mean Median Mode

In [11]:
# Use the NumPy mean() method to find the average speed:

import numpy

speed = [99,86,87,88,111,86,103,87,94,78,77,85,86]

x = numpy.mean(speed)

print(x)

89.76923076923077


In [12]:
# Use the NumPy median() method to find the middle value:

import numpy

speed = [99,86,87,88,111,86,103,87,94,78,77,85,86]

x = numpy.median(speed)

print(x)

87.0


In [13]:
# Use the SciPy mode() method to find the number that appears the most:

from scipy import stats

speed = [99,86,87,88,111,86,103,87,94,78,77,85,86]

x = stats.mode(speed)

print(x)

ModeResult(mode=86, count=3)


# Standard deviation is a number that describes how spread out the values are.

In [14]:
# A low standard deviation means that most of the numbers are close to the mean (average) value.



import numpy

speed = [86,87,88,86,87,85,86]

x = numpy.std(speed)

print(x)

0.9035079029052513


In [15]:
# A high standard deviation means that the values are spread out over a wider range.

import numpy

speed = [32,111,138,28,59,77,97]

x = numpy.std(speed)

print(x)

37.84501153334721


In [ ]:
# if you multiply the standard deviation by itself, you get the variance!

import numpy

speed = [32,111,138,28,59,77,97]

x = numpy.var(speed)

print(x)

In [ ]:
# Percentiles are used in statistics to give you a number that describes the value 
# that a given percent of the values are lower than.

import numpy

ages = [5,31,43,48,50,41,7,11,15,39,80,82,32,2,8,6,25,36,27,61,31]

x = numpy.percentile(ages, 75)

print(x)

In [16]:
# What is the age that 90% of the people are younger than?

import numpy

ages = [5,31,43,48,50,41,7,11,15,39,80,82,32,2,8,6,25,36,27,61,31]

x = numpy.percentile(ages, 90)

print(x)

61.0


In [24]:
# Log transformation: compress a long right tail

df = pd.read_csv('trip_sample.csv')

df['fare_log'] = np.log1p(df['fare_cad'])

df

,trip_id,mode,distance_km,fare_cad,wait_min,rating,peak,fare_log
0,T001,Bike,3.2,0.00,4.0,5,Yes,0.000000
1,T002,Taxi,8.4,19.50,6.0,4,No,3.020425
2,T003,Bus,6.1,3.75,9.0,3,Yes,1.558145
3,T004,Taxi,2.7,11.20,12.0,2,Yes,2.501436
4,T005,Bus,14.3,3.75,18.0,2,Yes,1.558145
5,T006,Taxi,5.1,13.20,NaN,4,No,2.653242


In [26]:
# Min–max scaling 

x = df['distance_km']

df['distance_01'] = (
    (x - x.min()) /
    (x.max() - x.min())
)

df[['distance_km','distance_01']].head()


,distance_km,distance_01
0,3.2,0.043103
1,8.4,0.491379
2,6.1,0.293103
3,2.7,0.000000
4,14.3,1.000000


In [27]:
# Standardization: read values in standard-deviation units

x = df['wait_min']

df['wait_z'] = (x - x.mean()) / x.std()

# or with scipy
from scipy.stats import zscore
df['wait_z2'] = zscore(x, nan_policy='omit')
    

In [28]:
df

,trip_id,mode,distance_km,fare_cad,wait_min,rating,peak,fare_log,distance_01,wait_z,wait_z2
0,T001,Bike,3.2,0.00,4.0,5,Yes,0.000000,0.043103,-1.055418,-1.179993
1,T002,Taxi,8.4,19.50,6.0,4,No,3.020425,0.491379,-0.691481,-0.773099
2,T003,Bus,6.1,3.75,9.0,3,Yes,1.558145,0.293103,-0.145575,-0.162758
3,T004,Taxi,2.7,11.20,12.0,2,Yes,2.501436,0.000000,0.400331,0.447584
4,T005,Bus,14.3,3.75,18.0,2,Yes,1.558145,1.000000,1.492143,1.668266
5,T006,Taxi,5.1,13.20,NaN,4,No,2.653242,0.206897,NaN,NaN


# Scale Features

In [17]:
# Scale all values in the Weight and Volume columns:

import pandas
from sklearn import linear_model
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()

df = pandas.read_csv("scale_data.csv")

X = df[['Weight', 'Volume']]

scaledX = scale.fit_transform(X)

print(scaledX)

[[-2.10389253 -1.59336644]
 [-0.55407235 -1.07190106]
 [-1.52166278 -1.59336644]
 [-1.78973979 -1.85409913]
 [-0.63784641 -0.28970299]
 [-1.52166278 -1.59336644]
 [-0.76769621 -0.55043568]
 [ 0.3046118  -0.28970299]
 [-0.7551301  -0.28970299]
 [-0.59595938 -0.0289703 ]
 [-1.30803892 -1.33263375]
 [-1.26615189 -0.81116837]
 [-0.7551301  -1.59336644]
 [-0.16871166 -0.0289703 ]
 [ 0.14125238 -0.0289703 ]
 [ 0.15800719 -0.0289703 ]
 [ 0.3046118  -0.0289703 ]
 [-0.05142797  1.53542584]
 [-0.72580918 -0.0289703 ]
 [ 0.14962979  1.01396046]
 [ 1.2219378  -0.0289703 ]
 [ 0.5685001   1.01396046]
 [ 0.3046118   1.27469315]
 [ 0.51404696 -0.0289703 ]
 [ 0.51404696  1.01396046]
 [ 0.72348212 -0.28970299]
 [ 0.8281997   1.01396046]
 [ 1.81254495  1.01396046]
 [ 0.96642691 -0.0289703 ]
 [ 1.72877089  1.01396046]
 [ 1.30990057  1.27469315]
 [ 1.90050772  1.01396046]
 [-0.23991961 -0.0289703 ]
 [ 0.40932938 -0.0289703 ]
 [ 0.47215993 -0.0289703 ]
 [ 0.4302729   2.31762392]]
